# Understanding Nested Functions, Closures, and List Behavior in Python

One of the most powerful — and often confusing — features of Python is the **ability to define a function inside another function**. At first glance it looks simple, but when mutable objects like lists are involved, surprising behavior appears: values seem to “remember” previous calls even after the outer function has finished executing.

This article explains how nested functions work, **why lists persist, and what Python is actually doing internally.**

---

## Nested Functions: The Basic Idea

Python allows functions to be defined inside other functions.

In [2]:
def outer():
    l=[]
    def inner(x):
        l.append(x)
        return l
    
    return inner

Here:

- `outer()` creates a list `l`
- `inner()` is defined inside `outer()`
- `inner()` uses the list `l`
- `outer()` returns the function `inner`

**This structure creates something much more powerful than a normal function.**

---

## What Happens During Execution?

### Let us run:
```python
f = outer()
```
### Step-by-step:

- `outer()` starts execution.
- A new empty list is created: `l = []`
- Python creates the function `inner`.
- `outer()` returns `inner`.

At this moment, `outer()` has finished execution.

A natural question arises:

**👉 Shouldn’t the list l be destroyed now?**

**Normally, yes. Local variables disappear when a function ends. But here, something special happens.**

---

## The Concept of a Closure

- The returned function `inner` still needs access to `l`.
- So Python keeps the **surrounding environment** alive.

This mechanism is called a **closure**.

A closure is:
> **A function that remembers variables from the scope in which it was created, even after that scope has finished execution.**

So the list `l` continues to exist because `inner` depends on it.

---

## Observing the List Behavior

Now call the returned function multiple times:

In [3]:
f = outer()

print(f(10))
print(f(20))
print(f(30))

[10]
[10, 20]
[10, 20, 30]


**The list keeps growing.**

**Why?**
> Because every call to `inner()` modifies the same list object created during the original call to `outer()`.

No new list is created.

---

## Mutable Objects Are the Key

Lists in Python are mutable, meaning they can be changed without creating a new object.

When we write:

```python
l.append(x)
```

we are modifying the existing list stored in the closure.

Conceptually:
```
inner  ─────►  l (single shared list)
```
***Each function call accesses the same memory location.***

---

## Python’s Scope Resolution (LEGB Rule)

When Python looks for a variable, it searches in this order:

- `Local` – inside the current function
- `Enclosing` – outer function scope
- `Global`
- `Built-in`

Inside `inner`, the variable `l` is not local, so Python finds it in the enclosing scope (`outer`).

---

## Modify vs Reassign — A Critical Difference

Consider this change:

In [4]:
def outer():
    l = []

    def inner(x):
        l = [x]  # NEW local variable!
        return l

    return inner

In [5]:
f = outer()

print(f(10))
print(f(20))
print(f(30))

[10]
[20]
[30]


**Why?**

Because:

- `l = [x]` creates a **new local variable**
- It no longer refers to the outer list

**Python treats assignment as creating a new local binding unless told otherwise.**

| Operation     | Effect                   |
| ------------- | ------------------------ |
| `l.append(x)` | modifies outer list     |
| `l += [x]`    | modifies outer list     |
| `l = [x]`     | creates new local list  |


---

## Using nonlocal for Reassignment

If we want to modify the outer variable itself:

In [6]:
def outer():
    l = []

    def inner(x):
        nonlocal l
        l = l + [x]
        return l

    return inner

In [7]:
f = outer()

print(f(10))
print(f(20))
print(f(30))

[10]
[10, 20]
[10, 20, 30]


The keyword nonlocal tells Python:

> “Use the variable from the enclosing scope.”

---



## Mental Model

The easiest way to understand closures:

* `outer()` creates **storage**
* `inner()` keeps **access** to that storage
* The storage survives as long as the inner function exists

Even though the outer function finishes, its data remains alive.

---

## Why This Feature Is Powerful

Closures allow functions to maintain internal state without using classes.

They are widely used in:

* Counters
* Stateful functions
* Caching mechanisms
* Function factories
* Decorators
* Functional programming patterns

Example counter:

```python
def counter():
    count = 0

    def inc():
        nonlocal count
        count += 1
        return count

    return inc
```

Each call remembers previous values.

---

## Closures vs Classes (Insight)

Closures behave very similarly to objects:

| Closure            | Class             |
| ------------------ | ----------------- |
| Enclosed variables | Object attributes |
| Inner function     | Methods           |
| Persistent state   | Instance state    |

In many cases, a closure is simply a lightweight object without explicitly defining a class.

---

## Conclusion

Nested functions in Python are more than just functions inside functions. They introduce the concept of **closures**, allowing inner functions to remember and modify variables from their enclosing scope.

When mutable objects like lists are involved, this leads to persistent behavior across function calls — not because Python is repeating work, but because the same object continues to live in memory.

Understanding this idea unlocks deeper Python concepts such as decorators, functional design patterns, and stateful computation — making it an essential topic for every serious Python programmer.


---